In [1]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

In [18]:
from typing import (
    Any, 
    Awaitable, 
    Callable, 
    Dict, 
    Optional, 
    Sequence,
    Union, 
    List
)
from llama_index.core.tools.types import BaseTool
from llama_index.core.base.llms.types import (
    ChatMessage,
    ChatResponse,
    ChatResponseAsyncGen,
    ChatResponseGen,
    CompletionResponse,
    CompletionResponseAsyncGen,
    CompletionResponseGen,
    LLMMetadata,
    MessageRole,
)
from llama_index.core.llms.function_calling import FunctionCallingLLM, ToolSelection
from llama_index.core.bridge.pydantic import Field
from llama_index.core.callbacks import CallbackManager
from llama_index.core.constants import DEFAULT_TEMPERATURE
from llama_index.core.llms.callbacks import llm_chat_callback, llm_completion_callback
from llama_index.core.base.llms.generic_utils import (
    achat_to_completion_decorator,
    acompletion_to_chat_decorator,
    astream_chat_to_completion_decorator,
    astream_completion_to_chat_decorator,
    chat_to_completion_decorator,
    completion_to_chat_decorator,
    stream_chat_to_completion_decorator,
    stream_completion_to_chat_decorator,
)
from llama_index.core.llms.llm import LLM
from llama_index.core.types import BaseOutputParser, PydanticProgramMode
from llama_index.llms.litellm.utils import (
    acompletion_with_retry,
    completion_with_retry,
    from_litellm_message,
    is_function_calling_model,
    openai_modelname_to_contextsize,
    to_openai_message_dicts,
    validate_litellm_api_key,
)

DEFAULT_LITELLM_MODEL = "gpt-3.5-turbo"

def force_single_tool_call(response: ChatResponse) -> None:
    tool_calls = response.message.additional_kwargs.get("tool_calls", [])
    if len(tool_calls) > 1:
        response.message.additional_kwargs["tool_calls"] = [tool_calls[0]]

class LiteLLM(FunctionCallingLLM):
    """LiteLLM.

    Examples:
        `pip install llama-index-llms-litellm`

        ```python
        import os
        from llama_index.core.llms import ChatMessage
        from llama_index.llms.litellm import LiteLLM

        # Set environment variables
        os.environ["OPENAI_API_KEY"] = "your-openai-api-key"
        os.environ["COHERE_API_KEY"] = "your-cohere-api-key"

        # Define a chat message
        message = ChatMessage(role="user", content="Hey! how's it going?")

        # Initialize LiteLLM with the desired model
        llm = LiteLLM(model="gpt-3.5-turbo")

        # Call the chat method with the message
        chat_response = llm.chat([message])

        # Print the response
        print(chat_response)
        ```
    """

    model: str = Field(
        default=DEFAULT_LITELLM_MODEL,
        description=(
            "The LiteLLM model to use. "
            "For complete list of providers https://docs.litellm.ai/docs/providers"
        ),
    )
    temperature: float = Field(
        default=DEFAULT_TEMPERATURE,
        description="The temperature to use during generation.",
        ge=0.0,
        le=1.0,
    )
    max_tokens: Optional[int] = Field(
        description="The maximum number of tokens to generate.",
        gt=0,
    )
    additional_kwargs: Dict[str, Any] = Field(
        default_factory=dict,
        description="Additional kwargs for the LLM API.",
        # for all inputs https://docs.litellm.ai/docs/completion/input
    )
    max_retries: int = Field(
        default=10, description="The maximum number of API retries."
    )

    def __init__(
        self,
        model: str = DEFAULT_LITELLM_MODEL,
        temperature: float = DEFAULT_TEMPERATURE,
        max_tokens: Optional[int] = None,
        additional_kwargs: Optional[Dict[str, Any]] = None,
        max_retries: int = 10,
        api_key: Optional[str] = None,
        api_type: Optional[str] = None,
        api_base: Optional[str] = None,
        callback_manager: Optional[CallbackManager] = None,
        system_prompt: Optional[str] = None,
        messages_to_prompt: Optional[Callable[[Sequence[ChatMessage]], str]] = None,
        completion_to_prompt: Optional[Callable[[str], str]] = None,
        pydantic_program_mode: PydanticProgramMode = PydanticProgramMode.DEFAULT,
        output_parser: Optional[BaseOutputParser] = None,
        **kwargs: Any,
    ) -> None:
        if "custom_llm_provider" in kwargs:
            if (
                kwargs["custom_llm_provider"] != "ollama"
                and kwargs["custom_llm_provider"] != "vllm"
            ):  # don't check keys for local models
                validate_litellm_api_key(api_key, api_type)
        else:  # by default assume it's a hosted endpoint
            validate_litellm_api_key(api_key, api_type)

        additional_kwargs = additional_kwargs or {}
        if api_key is not None:
            additional_kwargs["api_key"] = api_key
        if api_type is not None:
            additional_kwargs["api_type"] = api_type
        if api_base is not None:
            additional_kwargs["api_base"] = api_base

        super().__init__(
            model=model,
            temperature=temperature,
            max_tokens=max_tokens,
            additional_kwargs=additional_kwargs,
            max_retries=max_retries,
            callback_manager=callback_manager,
            system_prompt=system_prompt,
            messages_to_prompt=messages_to_prompt,
            completion_to_prompt=completion_to_prompt,
            pydantic_program_mode=pydantic_program_mode,
            output_parser=output_parser,
            **kwargs,
        )

    def _get_model_name(self) -> str:
        model_name = self.model
        if "ft-" in model_name:  # legacy fine-tuning
            model_name = model_name.split(":")[0]
        elif model_name.startswith("ft:"):
            model_name = model_name.split(":")[1]

        return model_name

    @classmethod
    def class_name(cls) -> str:
        return "litellm_llm"

    @property
    def metadata(self) -> LLMMetadata:
        return LLMMetadata(
            context_window=openai_modelname_to_contextsize(self._get_model_name()),
            num_output=self.max_tokens or -1,
            is_chat_model=True,
            is_function_calling_model=is_function_calling_model(self._get_model_name()),
            model_name=self.model,
        )

    @llm_chat_callback()
    def chat(self, messages: Sequence[ChatMessage], **kwargs: Any) -> ChatResponse:
        if self._is_chat_model:
            chat_fn = self._chat
        else:
            chat_fn = completion_to_chat_decorator(self._complete)
        return chat_fn(messages, **kwargs)

    @llm_chat_callback()
    def stream_chat(
        self, messages: Sequence[ChatMessage], **kwargs: Any
    ) -> ChatResponseGen:
        if self._is_chat_model:
            stream_chat_fn = self._stream_chat
        else:
            stream_chat_fn = stream_completion_to_chat_decorator(self._stream_complete)
        return stream_chat_fn(messages, **kwargs)

    @llm_completion_callback()
    def complete(
        self, prompt: str, formatted: bool = False, **kwargs: Any
    ) -> CompletionResponse:
        # litellm assumes all llms are chat llms
        if self._is_chat_model:
            complete_fn = chat_to_completion_decorator(self._chat)
        else:
            complete_fn = self._complete

        return complete_fn(prompt, **kwargs)

    @llm_completion_callback()
    def stream_complete(
        self, prompt: str, formatted: bool = False, **kwargs: Any
    ) -> CompletionResponseGen:
        if self._is_chat_model:
            stream_complete_fn = stream_chat_to_completion_decorator(self._stream_chat)
        else:
            stream_complete_fn = self._stream_complete
        return stream_complete_fn(prompt, **kwargs)

    @property
    def _is_chat_model(self) -> bool:
        # litellm assumes all llms are chat llms
        return True

    @property
    def _model_kwargs(self) -> Dict[str, Any]:
        base_kwargs = {
            "model": self.model,
            "temperature": self.temperature,
            "max_tokens": self.max_tokens,
        }
        return {
            **base_kwargs,
            **self.additional_kwargs,
        }

    def _get_all_kwargs(self, **kwargs: Any) -> Dict[str, Any]:
        return {
            **self._model_kwargs,
            **kwargs,
        }

    def _chat(self, messages: Sequence[ChatMessage], **kwargs: Any) -> ChatResponse:
        if not self._is_chat_model:
            raise ValueError("This model is not a chat model.")

        message_dicts = to_openai_message_dicts(messages)
        all_kwargs = self._get_all_kwargs(**kwargs)
        if "max_tokens" in all_kwargs and all_kwargs["max_tokens"] is None:
            all_kwargs.pop(
                "max_tokens"
            )  # don't send max_tokens == None, this throws errors for Non OpenAI providers

        response = completion_with_retry(
            is_chat_model=self._is_chat_model,
            max_retries=self.max_retries,
            messages=message_dicts,
            stream=False,
            **all_kwargs,
        )
        message_dict = response["choices"][0]["message"]
        message = from_litellm_message(message_dict)

        return ChatResponse(
            message=message,
            raw=response,
            additional_kwargs=self._get_response_token_counts(response),
        )

    def _stream_chat(
        self, messages: Sequence[ChatMessage], **kwargs: Any
    ) -> ChatResponseGen:
        if not self._is_chat_model:
            raise ValueError("This model is not a chat model.")

        message_dicts = to_openai_message_dicts(messages)
        all_kwargs = self._get_all_kwargs(**kwargs)
        if "max_tokens" in all_kwargs and all_kwargs["max_tokens"] is None:
            all_kwargs.pop(
                "max_tokens"
            )  # don't send max_tokens == None, this throws errors for Non OpenAI providers

        def gen() -> ChatResponseGen:
            content = ""
            function_call: Optional[dict] = None
            for response in completion_with_retry(
                is_chat_model=self._is_chat_model,
                max_retries=self.max_retries,
                messages=message_dicts,
                stream=True,
                **all_kwargs,
            ):
                delta = response["choices"][0]["delta"]
                role = delta.get("role") or MessageRole.ASSISTANT
                content_delta = delta.get("content", "") or ""
                content += content_delta

                function_call_delta = delta.get("function_call", None)
                if function_call_delta is not None:
                    if function_call is None:
                        function_call = function_call_delta

                        ## ensure we do not add a blank function call
                        if function_call.get("function_name", "") is None:
                            del function_call["function_name"]
                    else:
                        function_call["arguments"] += function_call_delta["arguments"]

                additional_kwargs = {}
                if function_call is not None:
                    additional_kwargs["function_call"] = function_call

                yield ChatResponse(
                    message=ChatMessage(
                        role=role,
                        content=content,
                        additional_kwargs=additional_kwargs,
                    ),
                    delta=content_delta,
                    raw=response,
                    additional_kwargs=self._get_response_token_counts(response),
                )

        return gen()

    def _complete(self, prompt: str, **kwargs: Any) -> CompletionResponse:
        raise NotImplementedError("litellm assumes all llms are chat llms.")

    def _stream_complete(self, prompt: str, **kwargs: Any) -> CompletionResponseGen:
        raise NotImplementedError("litellm assumes all llms are chat llms.")

    def _get_max_token_for_prompt(self, prompt: str) -> int:
        try:
            import tiktoken
        except ImportError:
            raise ImportError(
                "Please install tiktoken to use the max_tokens=None feature."
            )
        context_window = self.metadata.context_window
        try:
            encoding = tiktoken.encoding_for_model(self._get_model_name())
        except KeyError:
            encoding = encoding = tiktoken.get_encoding(
                "cl100k_base"
            )  # default to using cl10k_base
        tokens = encoding.encode(prompt)
        max_token = context_window - len(tokens)
        if max_token <= 0:
            raise ValueError(
                f"The prompt is too long for the model. "
                f"Please use a prompt that is less than {context_window} tokens."
            )
        return max_token

    def _get_response_token_counts(self, raw_response: Any) -> dict:
        """Get the token usage reported by the response."""
        if not isinstance(raw_response, dict):
            return {}

        usage = raw_response.get("usage", {})
        return {
            "prompt_tokens": usage.get("prompt_tokens", 0),
            "completion_tokens": usage.get("completion_tokens", 0),
            "total_tokens": usage.get("total_tokens", 0),
        }

    # ===== Async Endpoints =====
    @llm_chat_callback()
    async def achat(
        self,
        messages: Sequence[ChatMessage],
        **kwargs: Any,
    ) -> ChatResponse:
        achat_fn: Callable[..., Awaitable[ChatResponse]]
        if self._is_chat_model:
            achat_fn = self._achat
        else:
            achat_fn = acompletion_to_chat_decorator(self._acomplete)
        return await achat_fn(messages, **kwargs)

    @llm_chat_callback()
    async def astream_chat(
        self,
        messages: Sequence[ChatMessage],
        **kwargs: Any,
    ) -> ChatResponseAsyncGen:
        astream_chat_fn: Callable[..., Awaitable[ChatResponseAsyncGen]]
        if self._is_chat_model:
            astream_chat_fn = self._astream_chat
        else:
            astream_chat_fn = astream_completion_to_chat_decorator(
                self._astream_complete
            )
        return await astream_chat_fn(messages, **kwargs)

    @llm_completion_callback()
    async def acomplete(
        self, prompt: str, formatted: bool = False, **kwargs: Any
    ) -> CompletionResponse:
        if self._is_chat_model:
            acomplete_fn = achat_to_completion_decorator(self._achat)
        else:
            acomplete_fn = self._acomplete
        return await acomplete_fn(prompt, **kwargs)

    @llm_completion_callback()
    async def astream_complete(
        self, prompt: str, formatted: bool = False, **kwargs: Any
    ) -> CompletionResponseAsyncGen:
        if self._is_chat_model:
            astream_complete_fn = astream_chat_to_completion_decorator(
                self._astream_chat
            )
        else:
            astream_complete_fn = self._astream_complete
        return await astream_complete_fn(prompt, **kwargs)

    async def _achat(
        self, messages: Sequence[ChatMessage], **kwargs: Any
    ) -> ChatResponse:
        if not self._is_chat_model:
            raise ValueError("This model is not a chat model.")

        message_dicts = to_openai_message_dicts(messages)
        all_kwargs = self._get_all_kwargs(**kwargs)
        response = await acompletion_with_retry(
            is_chat_model=self._is_chat_model,
            max_retries=self.max_retries,
            messages=message_dicts,
            stream=False,
            **all_kwargs,
        )
        message_dict = response["choices"][0]["message"]
        message = from_litellm_message(message_dict)

        return ChatResponse(
            message=message,
            raw=response,
            additional_kwargs=self._get_response_token_counts(response),
        )

    async def _astream_chat(
        self, messages: Sequence[ChatMessage], **kwargs: Any
    ) -> ChatResponseAsyncGen:
        if not self._is_chat_model:
            raise ValueError("This model is not a chat model.")

        message_dicts = to_openai_message_dicts(messages)
        all_kwargs = self._get_all_kwargs(**kwargs)

        async def gen() -> ChatResponseAsyncGen:
            content = ""
            function_call: Optional[dict] = None
            async for response in await acompletion_with_retry(
                is_chat_model=self._is_chat_model,
                max_retries=self.max_retries,
                messages=message_dicts,
                stream=True,
                **all_kwargs,
            ):
                delta = response["choices"][0]["delta"]
                role = delta.get("role") or MessageRole.ASSISTANT
                content_delta = delta.get("content", "") or ""
                content += content_delta

                function_call_delta = delta.get("function_call", None)
                if function_call_delta is not None:
                    if function_call is None:
                        function_call = function_call_delta

                        ## ensure we do not add a blank function call
                        if function_call.get("function_name", "") is None:
                            del function_call["function_name"]
                    else:
                        function_call["arguments"] += function_call_delta["arguments"]

                additional_kwargs = {}
                if function_call is not None:
                    additional_kwargs["function_call"] = function_call

                yield ChatResponse(
                    message=ChatMessage(
                        role=role,
                        content=content,
                        additional_kwargs=additional_kwargs,
                    ),
                    delta=content_delta,
                    raw=response,
                    additional_kwargs=self._get_response_token_counts(response),
                )

        return gen()

    async def _acomplete(self, prompt: str, **kwargs: Any) -> CompletionResponse:
        raise NotImplementedError("litellm assumes all llms are chat llms.")

    async def _astream_complete(
        self, prompt: str, **kwargs: Any
    ) -> CompletionResponseAsyncGen:
        raise NotImplementedError("litellm assumes all llms are chat llms.")
    
    def _prepare_chat_with_tools(
        self,
        tools: List["BaseTool"],
        user_msg: Optional[Union[str, ChatMessage]] = None,
        chat_history: Optional[List[ChatMessage]] = None,
        verbose: bool = False,
        allow_parallel_tool_calls: bool = False,
        **kwargs: Any,
    ):
        if self.metadata.is_function_calling_model is False:
            raise NotImplementedError("LLM is not a function calling model")
        tool_specs = [
            tool.metadata.to_openai_tool(skip_length_check=True) for tool in tools
        ]
        if isinstance(user_msg, str):
            user_msg = ChatMessage(role=MessageRole.USER, content=user_msg)

        messages = chat_history or []
        if user_msg:
            messages.append(user_msg)

        return {
            "messages": messages,
            "tools": tool_specs or None,
        }

    def _validate_chat_with_tools_response(
        self,
        response: ChatResponse,
        tools: List["BaseTool"],
        allow_parallel_tool_calls: bool = False,
        **kwargs: Any,
    ) -> ChatResponse:
        """Validate the response from chat_with_tools."""
        if not allow_parallel_tool_calls:
            force_single_tool_call(response)

        return response
    
    def get_tool_calls_from_response(
        self,
        response: "ChatResponse",
        error_on_no_tool_call: bool = True,
    ) -> List[ToolSelection]:
        """Predict and call the tool."""
        tool_calls = response.message.additional_kwargs.get("tool_calls", [])

        if len(tool_calls) < 1:
            if error_on_no_tool_call:
                raise ValueError(
                    f"Expected at least one tool call, but got {len(tool_calls)} tool calls."
                )
            else:
                return []

        tool_selections = []
        for tool_call in tool_calls:
            argument_dict = tool_call["function"]["arguments"]

            tool_selections.append(
                ToolSelection(
                    # tool ids not provided by Ollama
                    tool_id=tool_call["function"]["name"],
                    tool_name=tool_call["function"]["name"],
                    tool_kwargs=argument_dict,
                )
            )

        return tool_selections

In [4]:
os.environ["GEMINI_API_KEY"] = os.getenv("GOOGLE_API_KEY")

In [19]:
llm = LiteLLM(model="gemini/gemini-1.5-flash")

In [20]:
llm.metadata.is_function_calling_model

True

In [21]:
llm.complete("Hi!")

CompletionResponse(text='Hi there! What can I do for you today? \n', additional_kwargs={}, raw=ModelResponse(id='chatcmpl-0750062b-01b7-4261-b142-d91fdb0f0c0c', choices=[Choices(finish_reason='stop', index=0, message=Message(content='Hi there! What can I do for you today? \n', role='assistant', tool_calls=None, function_call=None))], created=1727360613, model='gemini-1.5-flash', object='chat.completion', system_fingerprint=None, usage=Usage(completion_tokens=12, prompt_tokens=3, total_tokens=15, completion_tokens_details=None), vertex_ai_grounding_metadata=[], vertex_ai_safety_results=[[{'category': 'HARM_CATEGORY_SEXUALLY_EXPLICIT', 'probability': 'NEGLIGIBLE'}, {'category': 'HARM_CATEGORY_HATE_SPEECH', 'probability': 'NEGLIGIBLE'}, {'category': 'HARM_CATEGORY_HARASSMENT', 'probability': 'NEGLIGIBLE'}, {'category': 'HARM_CATEGORY_DANGEROUS_CONTENT', 'probability': 'NEGLIGIBLE'}]], vertex_ai_citation_metadata=[]), logprobs=None, delta=None)

In [22]:
from llama_index.core.agent import FunctionCallingAgent
from llama_index.core.tools import FunctionTool

def multiply(a: int, b: int) -> int:
    """Multiple two integers and returns the result integer"""
    return a * b


multiply_tool = FunctionTool.from_defaults(fn=multiply)

def add(a: int, b: int) -> int:
    """Add two integers and returns the result integer"""
    return a + b


add_tool = FunctionTool.from_defaults(fn=add)

In [23]:
tools = [add_tool, multiply_tool]

In [27]:
llm._prepare_chat_with_tools(
    tools = tools,
    user_msg = "What's 5+5?"
)

{'messages': [ChatMessage(role=<MessageRole.USER: 'user'>, content="What's 5+5?", additional_kwargs={})],
 'tools': [{'type': 'function',
   'function': {'name': 'add',
    'description': 'add(a: int, b: int) -> int\nAdd two integers and returns the result integer',
    'parameters': {'properties': {'a': {'title': 'A', 'type': 'integer'},
      'b': {'title': 'B', 'type': 'integer'}},
     'required': ['a', 'b'],
     'type': 'object'}}},
  {'type': 'function',
   'function': {'name': 'multiply',
    'description': 'multiply(a: int, b: int) -> int\nMultiple two integers and returns the result integer',
    'parameters': {'properties': {'a': {'title': 'A', 'type': 'integer'},
      'b': {'title': 'B', 'type': 'integer'}},
     'required': ['a', 'b'],
     'type': 'object'}}}]}

In [29]:
import litellm
from litellm import completion

litellm.set_verbose=True

response = completion(
    model="gemini/gemini-1.5-flash",
    messages = [
        {
            "role": "user",
            "content": "What is 5*5?"
        }
    ],
    tools = [
        {
            'type': 'function',
            'function': {
                'name': 'add',
                'description': 'add(a: int, b: int) -> int\nAdd two integers and returns the result integer',
                'parameters': {
                    'properties': {
                        'a': {'title': 'A', 'type': 'integer'},
                        'b': {'title': 'B', 'type': 'integer'}},
                    'required': ['a', 'b'],
                    'type': 'object'}}},
        {
            'type': 'function',
            'function': {
                'name': 'multiply',
                'description': 'multiply(a: int, b: int) -> int\nMultiple two integers and returns the result integer',
                'parameters': {
                    'properties': {
                        'a': {'title': 'A', 'type': 'integer'},
                        'b': {'title': 'B', 'type': 'integer'}},
                    'required': ['a', 'b'],
                    'type': 'object'}}}]
)

22:35:53 - LiteLLM:WARNING: utils.py:351 - `litellm.set_verbose` is deprecated. Please set `os.environ['LITELLM_LOG'] = 'DEBUG'` for debug logs.




Request to litellm:
litellm.completion(model='gemini/gemini-1.5-flash', messages=[{'role': 'user', 'content': 'What is 5*5?'}], tools=[{'type': 'function', 'function': {'name': 'add', 'description': 'add(a: int, b: int) -> int\nAdd two integers and returns the result integer', 'parameters': {'properties': {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}, 'required': ['a', 'b'], 'type': 'object'}}}, {'type': 'function', 'function': {'name': 'multiply', 'description': 'multiply(a: int, b: int) -> int\nMultiple two integers and returns the result integer', 'parameters': {'properties': {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}, 'required': ['a', 'b'], 'type': 'object'}}}])


SYNC kwargs[caching]: False; litellm.cache: None; kwargs.get('cache')['no-cache']: False
Final returned optional params: {'tools': [{'function_declarations': [{'name': 'add', 'description': 'add(a: int, b: int) -> int\nAdd two integers and returns

BadRequestError: litellm.BadRequestError: VertexAIException BadRequestError - {
  "error": {
    "code": 400,
    "message": "Invalid JSON payload received. Unknown name \"title\" at 'tools[0].function_declarations[0].parameters.properties[0].value': Cannot find field.\nInvalid JSON payload received. Unknown name \"title\" at 'tools[0].function_declarations[0].parameters.properties[1].value': Cannot find field.\nInvalid JSON payload received. Unknown name \"title\" at 'tools[0].function_declarations[1].parameters.properties[0].value': Cannot find field.\nInvalid JSON payload received. Unknown name \"title\" at 'tools[0].function_declarations[1].parameters.properties[1].value': Cannot find field.",
    "status": "INVALID_ARGUMENT",
    "details": [
      {
        "@type": "type.googleapis.com/google.rpc.BadRequest",
        "fieldViolations": [
          {
            "field": "tools[0].function_declarations[0].parameters.properties[0].value",
            "description": "Invalid JSON p

In [30]:
from llama_index.core.agent import FunctionCallingAgent

llm = LiteLLM(model="gpt-4o-mini")

agent = FunctionCallingAgent.from_tools(
    [multiply_tool, add_tool],
    llm=llm,
    verbose=True,
    allow_parallel_tool_calls=False,
)

In [31]:
response = agent.chat("What is (121 + 2) * 5?")
print(str(response))

22:36:55 - LiteLLM:WARNING: utils.py:351 - `litellm.set_verbose` is deprecated. Please set `os.environ['LITELLM_LOG'] = 'DEBUG'` for debug logs.


> Running step ea2783ba-23ed-4e33-8f6e-c2c12b410e8e. Step input: What is (121 + 2) * 5?
Added user message to memory: What is (121 + 2) * 5?


Request to litellm:
litellm.completion(messages=[{'role': <MessageRole.USER: 'user'>, 'content': 'What is (121 + 2) * 5?'}], stream=False, model='gpt-4o-mini', temperature=0.1, tools=[{'type': 'function', 'function': {'name': 'multiply', 'description': 'multiply(a: int, b: int) -> int\nMultiple two integers and returns the result integer', 'parameters': {'properties': {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}, 'required': ['a', 'b'], 'type': 'object'}}}, {'type': 'function', 'function': {'name': 'add', 'description': 'add(a: int, b: int) -> int\nAdd two integers and returns the result integer', 'parameters': {'properties': {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}, 'required': ['a', 'b'], 'type': 'object'}}}])


SYNC kwargs[caching]: False; litellm.cache: None; kwargs.